# Functions

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* write functions that you can **check** (`assert` from the first line);
* understand **scope**: which variables a function sees and which it changes;
* use functions as **objects**: pass them, store them, return them;
* master the **arguments**: defaults, keywords, `*args`, `**kwargs`, type hints.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## 1. Anatomy of a function

A function is a named piece of code: it takes **arguments**, does some work and
**returns** a result. This one returns the first `n` Fibonacci numbers:

In [ ]:
def fib(n):
    """Return a list with the first n Fibonacci numbers."""
    ns = [0, 1]
    for i in range(2, n):
        ns.append(ns[i - 2] + ns[i - 1])
    return ns

In [ ]:
print(fib(10))

* `def` + **name** + **arguments**: the *interface*. It is what the user sees.
* The **docstring** says what the function does. `help(fib)` shows it.
* The indented block is the **body**. `return` ends the function and gives back a value
  (without `return`, the function returns `None`).
* A function is also an object, of type `function`.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Does fib do what its docstring says?</b></p>

The docstring promises *the first n numbers*. **Before running anything**, what do
`fib(1)` and `fib(0)` return? Is that what the docstring promises?

</div>

In [ ]:
print(fib(1), fib(0))

### 1.1 assert: write down what you expect

`assert condition` does nothing if the condition is true, and stops the program with
an `AssertionError` if it is false. It is the simplest **test**:

In [ ]:
assert fib(3) == [0, 1, 1]
assert len(fib(10)) == 10

In [ ]:
assert len(fib(1)) == 1, 'fib(1) must have one element'

The test found the bug that the eye missed. From now on: **every function comes with its
asserts**. In [Testing](./about_testing.ipynb) we will turn them into proper tests.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Fix fib</b></p>

1. Fix `fib` so that `len(fib(n)) == n` for every `n >= 0`.
2. Write the asserts that prove it: `n = 0, 1, 2` and a large `n`.
3. What should `fib(-1)` do? Decide, and make it do it.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def fib(n):
    """Return a list with the first n Fibonacci numbers."""
    if n < 0:
        raise ValueError(f'n must be non-negative, got {n}')
    ns = [0, 1]
    for i in range(2, n):
        ns.append(ns[i - 2] + ns[i - 1])
    return ns[:n]

assert fib(0) == []
assert fib(1) == [0]
assert fib(2) == [0, 1]
assert len(fib(50)) == 50
```

For `n < 0` there are two reasonable answers: return `[]` or raise an error. A negative
length is almost certainly a mistake of the caller, and *errors should never pass
silently*: raising is the better choice.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Mean and variance</b></p>

Write a function `mean_var(xs)` that returns the mean and the variance of a list of
numbers, without numpy. Check it **against numpy** with `assert` and `math.isclose`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def mean_var(xs):
    """Return the mean and the (population) variance of a list of numbers."""
    n = len(xs)
    mean = sum(xs) / n
    var = sum((x - mean) ** 2 for x in xs) / n
    return mean, var

np.random.seed(1)
xs = list(np.random.normal(3., 2., 1000))
mean, var = mean_var(xs)
assert math.isclose(mean, np.mean(xs))
assert math.isclose(var, np.var(xs))
```

Careful: `np.var` divides by `n` by default; `np.var(xs, ddof=1)` divides by `n - 1`.
Checking against a reference forces you to know what the reference computes.

</details>

## 2. Scope

The variables created inside a function are **local**: they live in the function's own
**scope** and disappear when it returns.

Python looks for a name in this order (**LEGB**): **L**ocal, **E**nclosing function,
**G**lobal (the module or notebook), **B**uilt-in.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — What is printed?</b></p>

**Decide before running**: what do the two `print` show?

```python
x = 10

def f():
    x = 5
    return x

print(f())
print(x)
```

</div>

In [ ]:
x = 10

def f():
    x = 5
    return x

print(f())
print(x)

`x = 5` creates a **new local** `x` inside `f`. The global `x` is not touched.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — And now?</b></p>

This one *reads* `x` and then assigns it. What happens, and why?

```python
def g():
    print(x)
    x = 5
```

</div>

In [ ]:
def g():
    print(x)
    x = 5

g()

<details>
<summary><b>Solution</b></summary>

`UnboundLocalError`. Python decides that a name is local **when it compiles the function**:
since `x` is assigned somewhere in `g`, `x` is local in the *whole* body, also in the
`print` line, where it has no value yet. The global `x` is never looked up.

</details>

***[+] Lookout.*** `global x` and `nonlocal x` let a function rebind a name of the global or of the
enclosing scope. They exist, but they are rarely a good idea: pass arguments and return
results instead.

### 2.1 Side effects

A function can **change** a mutable argument (a list, a dict, an array), whether or not
it returns something:

In [ ]:
def normalize(xs):
    total = sum(xs)
    for i in range(len(xs)):
        xs[i] = xs[i] / total
    return xs

weights = [1., 2., 1.]
probs = normalize(weights)
print(probs)
print(weights)

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Is this a good function?</b></p>

`normalize` changed `weights`. Is that a bug or a feature? When would you want it?
How would you write a version that leaves its argument untouched?

</div>

<details>
<summary><b>Solution</b></summary>

Both designs exist: `list.sort()` changes the list in place and returns `None`;
`sorted(xs)` returns a new list. The bad design is the **mix**: changing the argument
*and* returning it, so the user believes they got a copy.

Rule of thumb for this course: **functions do not modify their arguments**. They return
new objects:

```python
def normalize(xs):
    total = sum(xs)
    return [x / total for x in xs]
```

</details>

## 3. Recursion

A function can call itself. The factorial, $n! = n \, (n-1)!$, with $0! = 1$:

In [ ]:
def factorial(n):
    """Return n! = n (n-1) ... 1."""
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))

Every recursion needs a **base case** that stops it (`n <= 1`), and each call must
come closer to it.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Validation</b></p>

What does `factorial(-3)` return? And `factorial(2.5)`? Are they correct?
Add the checks that make the function refuse them.

</div>

<details>
<summary><b>Solution</b></summary>

Both return a number, `1` and `2.5 * 1.5 * 1 = 3.75`, and both are wrong: $n!$ is defined
for non-negative integers only. A wrong answer without an error is the worst kind of bug.

```python
def factorial(n):
    if not isinstance(n, int) or n < 0:
        raise ValueError(f'factorial needs a non-negative integer, got {n!r}')
    return 1 if n <= 1 else n * factorial(n - 1)
```

</details>

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Recursion or loop?</b></p>

`factorial(5000)` fails with a recursive version and works with a loop. **Before
running it**: why?

</div>

In [ ]:
try:
    factorial(5000)
except RecursionError as e:
    print(type(e).__name__, ':', e)

<details>
<summary><b>Solution</b></summary>

Each call waits for the next one, and Python keeps all the pending calls in memory (the
*call stack*). Python limits its depth: `sys.getrecursionlimit()` is 1000 by default.
A loop has no such limit:

```python
def factorial_loop(n):
    result = 1
    for k in range(2, n + 1):
        result *= k
    return result

assert factorial_loop(5000) == math.factorial(5000)
```
 In Python, recursion is elegant for problems that are naturally recursive
(trees, divide and conquer), but a loop is usually safer and faster.

</details>

## 4. Functions are objects

A function is an object like a number or a list: you can **assign** it to a variable,
**pass** it as an argument, store it in a list, and **return** it from another function.

In [ ]:
f = factorial
print(f(4), type(f), f.__name__)

### 4.1 Functions as arguments

This is how physics code is written: a function that **integrates**, **derives** or
**fits** takes the function to work on as an argument.

In [ ]:
def integrate(f, a, b, n=1000):
    """Integral of f over [a, b] with the midpoint rule and n intervals."""
    h = (b - a) / n
    total = 0.
    for i in range(n):
        total += f(a + (i + 0.5) * h)
    return h * total

print(integrate(math.sin, 0., math.pi))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A numerical derivative</b></p>

Write `derivative(f, x, h=1e-5)` that computes $f'(x)$ with the central difference
$[f(x+h) - f(x-h)] / 2h$. Test it with `math.sin` (the derivative is `math.cos`) at
several points, using `math.isclose` with a sensible tolerance.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def derivative(f, x, h=1e-5):
    """Central-difference derivative of f at x."""
    return (f(x + h) - f(x - h)) / (2. * h)

for x in [0., 0.5, 1., 3.]:
    assert math.isclose(derivative(math.sin, x), math.cos(x), abs_tol=1e-8)
```

The error of the central difference goes as $h^2$, but if $h$ is too small the rounding
error grows as $\epsilon / h$. Try `h=1e-12`: the test fails. The tolerance of a test
is part of the physics, not a detail.

</details>

### 4.2 Functions inside functions

A function can be defined inside another one. It is **local**: it only exists inside.

In [ ]:
def closest_to(x0, xs):
    """Return the element of xs closest to x0."""
    def distance(x0, xi):
        return abs(x0 - xi)

    d, x = 1e16, 0
    for xi in xs:
        di = distance(x0, xi)
        if di < d:
            d, x = di, xi
    return x

print(closest_to(2j, [0., 0.5, 1j, 20.]))

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Three weak points</b></p>

`closest_to` works for the example. Find three situations where it **silently** gives a
wrong or questionable answer.

</div>

<details>
<summary><b>Solution</b></summary>

1. `xs` empty: it returns `0`, an element that is not in the list.
2. All elements farther than `1e16` (e.g. `xs = [1e20]`): it returns `0` too.
   The *magic number* `1e16` is an arbitrary limit.
3. Ties: it returns the first one, silently. That may be fine, but the docstring
   should say it.

The idiomatic version has no magic number and fails loudly on an empty list:

```python
def closest_to(x0, xs):
    return min(xs, key=lambda xi: abs(x0 - xi))
```

`min` takes a function as argument, `key`: functions as objects again.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Closest point to the origin</b></p>

Given a list of points `[(x, y), ...]`, write a function that returns the point
closest to the origin and its distance. Use `min` with a `key`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def closest_to_origin(points):
    """Return the point (x, y) closest to the origin, and its distance."""
    p = min(points, key=lambda p: math.hypot(*p))
    return p, math.hypot(*p)

p, d = closest_to_origin([(3., 4.), (1., 1.), (-2., 0.)])
assert p == (1., 1.) and math.isclose(d, math.sqrt(2.))
```

</details>

### 4.3 Functions that return functions: closures

A function can **build** and return another function. The inner function remembers the
variables of the outer one even after it has returned: it is a **closure**.

`polynomial` takes the coefficients $[a_0, a_1, \dots, a_n]$ and returns the function
$p(x) = \sum_{i=0}^n a_i \, x^i$:

In [ ]:
def polynomial(coefficients):
    """Return the polynomial function with the given coefficients a0, a1, ..."""
    def p(x):
        y = 0.
        for i, a in enumerate(coefficients):
            y = y + a * x ** i
        return y
    return p

p = polynomial([1., -1., 2.])     # p(x) = 1 - x + 2 x^2
print(p(2.))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Will it work with an array?</b></p>

`p` was written thinking of a number `x`. **Decide**: does `p(np.linspace(-5, 5, 100))`
work, fail, or give something unexpected?

</div>

In [ ]:
xs = np.linspace(-5., 5., 100)
plt.plot(xs, p(xs))
plt.xlabel('x')
plt.ylabel('p(x)');

It works: `+`, `*` and `**` act element by element on numpy arrays. Python does not ask
for types; it only asks that the operations in the body make sense
(*duck typing*).

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A Poisson distribution, frozen</b></p>

1. Write `poisson(n, mu)`, the probability $P(n \mid \mu) = \mu^n e^{-\mu} / n!$.
2. Write `poisson_with(mu)`, which returns a function `f(n)` with $f(n) = P(n \mid \mu)$.
3. Check that $\sum_n f(n) \simeq 1$ and that the mean $\sum_n n f(n) \simeq \mu$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def poisson(n, mu):
    """Poisson probability of n counts with mean mu."""
    return mu ** n * math.exp(-mu) / math.factorial(n)

def poisson_with(mu):
    """Return the Poisson probability function of mean mu."""
    def f(n):
        return poisson(n, mu)
    return f

f = poisson_with(3.5)
ns = range(100)
assert math.isclose(sum(f(n) for n in ns), 1.)
assert math.isclose(sum(n * f(n) for n in ns), 3.5)
```

This is what `scipy.stats.poisson(3.5).pmf` does: a distribution with its parameter
*frozen*.

</details>

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Three functions</b></p>

We want three functions: $x$, $2x$ and $3x$. What does the last line print?

```python
fs = []
for k in [1, 2, 3]:
    fs.append(lambda x: k * x)

print(fs[0](1), fs[1](1), fs[2](1))
```

</div>

In [ ]:
fs = []
for k in [1, 2, 3]:
    fs.append(lambda x: k * x)

print(fs[0](1), fs[1](1), fs[2](1))

<details>
<summary><b>Solution</b></summary>

`3 3 3`. The closure remembers the **variable** `k`, not its value when the function was
created. When the functions are called, the loop has finished and `k` is `3`.
The fix is to freeze the value as a default argument:

```python
fs.append(lambda x, k=k: k * x)
```

</details>

## 5. lambda, map and filter

`lambda` defines a short function in one expression, without a name:

In [ ]:
square = lambda x: x * x
print(square(3.))

It is useful where a function is needed only once, as an argument. The built-ins
`map(f, xs)` and `filter(cond, xs)` apply a function to every element, or keep the
elements that satisfy a condition:

In [ ]:
ns = list(range(6))
squares = list(map(lambda n: n * n, ns))
evens = list(filter(lambda n: n % 2 == 0, ns))
print(ns, squares, evens, sep='\n')

In [ ]:
particles = [('e', 0.511), ('mu', 105.66), ('pi', 139.57), ('p', 938.27)]
print(sorted(particles, key=lambda p: p[1], reverse=True))

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Three ways to say the same thing</b></p>

```python
squares = []
for n in ns:
    squares.append(n * n)

squares = list(map(lambda n: n * n, ns))

squares = [n * n for n in ns]
```

Which one is the most readable? Which one would you use? (The third one is a
*comprehension*: it is the subject of the next chapter.)

</div>

*Style.* PEP 8 discourages `square = lambda x: x * x`: if the function has a name, use
`def`. Keep `lambda` for arguments, as in `key=lambda p: p[1]`.

***[+] Lookout.*** `functools.reduce` folds a list into one value, and the module `operator` has
the operators as functions (`operator.mul`, `operator.itemgetter(1)`…).
`functools.reduce(operator.mul, range(1, 6))` is $5!$.

## 6. Arguments

### 6.1 Default values

An argument can have a **default value**. If the caller does not give it, the default is
used. Arguments with defaults go after the ones without.

In [ ]:
def gaussian(x, mu=0., sigma=1.):
    """Normal probability density at x."""
    return math.exp(-0.5 * ((x - mu) / sigma) ** 2) / (math.sqrt(2. * math.pi) * sigma)

print(gaussian(0.), gaussian(1., mu=1.), gaussian(1., 0., 2.))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — The mutable default</b></p>

**Before running**: what do the three calls print?

```python
def add_hit(hit, hits=[]):
    hits.append(hit)
    return hits

print(add_hit(1))
print(add_hit(2))
print(add_hit(3, []))
```

</div>

In [ ]:
def add_hit(hit, hits=[]):
    hits.append(hit)
    return hits

print(add_hit(1))
print(add_hit(2))
print(add_hit(3, []))

<details>
<summary><b>Solution</b></summary>

`[1]`, `[1, 2]` and `[3]`. The default value is created **once**, when the function is
defined, not at every call: all the calls without `hits` share the same list.
Never use a mutable object (list, dict, array) as a default. The standard idiom:

```python
def add_hit(hit, hits=None):
    if hits is None:
        hits = []
    hits.append(hit)
    return hits
```

</details>

### 6.2 Positional and keyword arguments

Arguments can be passed by **position** or by **name** (keyword). By name, the order does
not matter, and the call explains itself:

In [ ]:
print(gaussian(1., 0., 2.))
print(gaussian(1., sigma=2., mu=0.))

### 6.3 Unpacking: * and **

`*` unpacks a list or a tuple into positional arguments; `**` unpacks a dictionary into
keyword arguments (keys = names of the arguments).

In [ ]:
point = (1., 0., 2.)
print(gaussian(*point))

pars = {'mu': 0., 'sigma': 2.}
print(gaussian(1., **pars))

### 6.4 Any number of arguments: *args and **kwargs

In the definition, `*args` collects the extra positional arguments in a tuple, and
`**kwargs` the extra keyword arguments in a dictionary.

In [ ]:
def report(name, *values, **options):
    print('name   :', name)
    print('values :', values)
    print('options:', options)

report('energy', 1.2, 3.4, 5.6, unit='GeV', error=0.1)

This is how `print(*objects, sep=' ', end='\n')` works. It is also the way to **pass
arguments through** a function without knowing them: we will need it in section 7.

### 6.5 Returning several values

A function returns one object, but it can be a **tuple**, which can be unpacked:

In [ ]:
def sum_and_difference(a, b):
    return a + b, a - b

result = sum_and_difference(4., 2.)
print(type(result), result)

total, delta = sum_and_difference(4., 2.)
print(total, delta)

### 6.6 Type hints

Python does not check types. You can still **annotate** the arguments and the result:
it documents the function, and tools such as editors or `mypy` use it.

In [ ]:
from collections.abc import Callable, Sequence

def polynomial(coefficients: Sequence[float]) -> Callable[[float], float]:
    """Return the polynomial function with the given coefficients a0, a1, ..."""
    def p(x: float) -> float:
        y = 0.
        for i, a in enumerate(coefficients):
            y = y + a * x ** i
        return y
    return p

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Are the hints enforced?</b></p>

What happens with `polynomial([1, 1])(np.array([1, 2]))`, and with
`polynomial('ab')(2)`? Run them and explain.

</div>

In [ ]:
print(polynomial([1, 1])(np.array([1, 2])))

In [ ]:
print(polynomial('ab')(2))

<details>
<summary><b>Solution</b></summary>

The first one works, although an array is not a `float`: hints are **not** checked when
the code runs. The second one fails, but not because of the hint: `'a' * 2 ** 0` is
`'a'`, and the error comes when Python tries `0. + 'a'`. The message talks about adding a
`float` and a `str`, far from the real cause. Hints help you and your tools *before*
running the code; they do not protect it at run time.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — A straight-line fit, in pairs</b></p>

Fit points $(x_i, y_i)$ with errors $\sigma_i$ to a line $y = a + b\,x$ by least squares.

* **A** writes `fit_line(xs, ys, sigmas)`, returning `a`, `b` and their errors.
* **B**, at the same time and without seeing A's code, writes the **tests**: generate
  points on a known line, with and without noise, and decide what must come out.
* Swap: run B's tests on A's code. Who found what?

This is step 5 of the [AI cycle](./python_intro.ipynb), with a classmate instead of an AI.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def fit_line(xs, ys, sigmas):
    """Weighted least-squares fit to y = a + b x. Return a, b, sigma_a, sigma_b."""
    w = [1. / s ** 2 for s in sigmas]
    S = sum(w)
    Sx = sum(wi * x for wi, x in zip(w, xs))
    Sy = sum(wi * y for wi, y in zip(w, ys))
    Sxx = sum(wi * x * x for wi, x in zip(w, xs))
    Sxy = sum(wi * x * y for wi, x, y in zip(w, xs, ys))
    D = S * Sxx - Sx ** 2
    a = (Sxx * Sy - Sx * Sxy) / D
    b = (S * Sxy - Sx * Sy) / D
    return a, b, math.sqrt(Sxx / D), math.sqrt(S / D)

# test: exact points on y = 1 + 2 x
xs = [0., 1., 2., 3.]
a, b, *_ = fit_line(xs, [1. + 2. * x for x in xs], [0.1] * 4)
assert math.isclose(a, 1.) and math.isclose(b, 2.)
```

Good tests also check: noisy points give `b` within a few errors of the true value;
two points give an exact line; all the `x` equal makes `D = 0` (what should happen?).

</details>

## 7. Decorators

With closures and `*args, **kwargs` we can write a function that takes a function and
returns an **improved** version of it. This one counts how many times a function is
called:

In [ ]:
import functools

def count_calls(f):
    @functools.wraps(f)          # keep the name and the docstring of f
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return f(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

`@count_calls` before a `def` is a shortcut for `fibo = count_calls(fibo)`:

In [ ]:
@count_calls
def fibo(n):
    """n-th Fibonacci number, recursive."""
    return n if n < 2 else fibo(n - 1) + fibo(n - 2)

print(fibo(20), fibo.calls)

Inside `fibo`, the name `fibo` is the decorated one: that is why the recursive calls are
counted too.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Twenty thousand calls</b></p>

`fibo(20)` needs more than 20 000 calls. Why so many? How do the calls grow with `n`?

</div>

<details>
<summary><b>Solution</b></summary>

Each call makes two more, and the same values are recomputed again and again:
`fibo(18)` is computed twice, `fibo(17)` three times… The number of calls grows like the
Fibonacci numbers themselves, exponentially, as $\phi^n$ with $\phi \simeq 1.618$.

</details>

***[+] Lookout.*** `functools.lru_cache` is a decorator that remembers the results already computed
(*memoization*). Put `@functools.lru_cache` **between** `@count_calls` and `def fibo`,
and count again: 39 calls instead of 21 891.
Decorators are everywhere: `@pytest.fixture`, `@property`, `@dataclass`, `@numba.jit`.

## 8. Summary

<details>
<summary><b>[i] Cheat sheet — Functions</b></summary>

| | |
|:--|:--|
| `def f(a, b=1, *args, **kwargs):` | definition: positional, default, extra positional, extra keyword |
| `"""docstring"""` | first line of the body; `help(f)` shows it |
| `assert cond, 'message'` | the simplest test |
| `f(1, b=2)`, `f(*xs)`, `f(**d)` | positional, keyword, unpacked list, unpacked dict |
| LEGB | Local, Enclosing, Global, Built-in |
| `lambda x: x * x` | anonymous function, for arguments |
| `map`, `filter`, `sorted(key=)`, `min(key=)` | functions that take functions |
| closure | inner function that remembers the outer variables |
| `@decorator` | `f = decorator(f)` |

**Traps**: mutable default arguments; late binding in closures; functions that modify
their arguments; recursion depth.

</details>

### Check yourself

1. What does a function without `return` return?
2. Why does `def g(): print(x); x = 5` fail, if `x` is defined outside?
3. Why is `def f(xs=[])` a bad idea? Write the correct idiom.
4. What is a closure? Give an example from physics.
5. What does `@count_calls` do to the function below it?

<details>
<summary><b>Solutions</b></summary>

1. `None`.
2. `x` is assigned inside `g`, so it is local in the whole body; at the `print` it has no
   value yet (`UnboundLocalError`).
3. The default list is created once and shared by all the calls. Use `xs=None` and
   `if xs is None: xs = []`.
4. A function built by another function, which keeps the outer variables: e.g. a
   Poisson distribution with $\mu$ frozen, or a polynomial with its coefficients.
5. It replaces it by `count_calls(f)`, a wrapper that counts the calls and then calls `f`.

</details>